In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

In [ ]:
# 1. 압축 해제
!unzip -q processed_data.zip -d /content/processed_data

# 2. 데이터 로드 확인
from datasets import load_from_disk
dataset = load_from_disk("/content/processed_data")
print(f"데이터 건수: {len(dataset)}")
print(dataset[0]) # 첫 번째 한국어 데이터가 잘 나오는지 확인

In [ ]:
import torch
import gc
from datasets import load_from_disk
try:
    dataset = load_from_disk("/content/processed_data")
except:
    dataset = load_from_disk("./processed_data")

from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig,
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, PeftModel
# from trl import SFTTrainer, SFTConfig # SFTConfig 추가

# 1. 기존 메모리 찌꺼기 제거 (코드 내에서도 수행)
gc.collect()
torch.cuda.empty_cache()

# 모델 ID (한국어 특화 Llama-3)
model_id = "beomi/Llama-3-Open-Ko-8B"

# 1. 4비트 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True, # 추가: 메모리 추가 절약
)


# 2. 모델 및 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map={"": 0} # 수정: 모델 전체를 GPU 0번에 강제 할당
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 3. LoRA 설정
peft_config = LoraConfig(
    r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 기존 SFTConfig 대신 일반 TrainingArguments 사용
training_args = TrainingArguments(
    output_dir="./ko-commerce-results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=5,
    fp16=True,
    save_strategy="no",
    gradient_checkpointing=True,
    report_to="none"
)

# 데이터 콜레이터 설정 (텍스트를 모델이 읽기 좋게 정렬)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# [핵심] SFTTrainer 대신 원본 'Trainer' 직접 사용 (에러 원천 차단)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset.map(lambda x: tokenizer(x["text"], truncation=True, max_length=512), batched=True),
    data_collator=data_collator,
)

print("🚀 라이브러리 충돌을 우회하여 학습을 시작합니다...")
trainer.train()

# 4. 어댑터 저장
model.save_pretrained("./final_adapter")
print("✅ 학습이 완료되었습니다!")